# Model Evaluation

In [ ]:
import sys
sys.path.append("..")

import importlib.util
import inspect
import pickle
import sys
from pathlib import Path

from config import medium_dataset_path

import numpy as np
import torch
from scipy.interpolate import PchipInterpolator
from scipy.optimize import linear_sum_assignment
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def find_repo_root():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / 'src').is_dir() and (path / 'config.py').is_file():
            return path
    raise RuntimeError('Repository root not found')

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

RUN_DIR = REPO_ROOT / 'runs/run_20260903_170728'
CHECKPOINT_NAME = 'best_checkpoint.pth'
DATASET_PATH = medium_dataset_path
NUMBER_OF_SAMPLES = 1000
BATCH_SIZE = 512
MODEL_BATCH_SIZE = 256
NB_OF_SAMPLING = 10
NB_OF_STEPS = None
SEED = 0
DEVICE = None
ALIGN_TO_TARGET = True
OUTPUT_PATH = REPO_ROOT / f"evaluation_results/run_20260903_170728_best_1000x10_{'target' if ALIGN_TO_TARGET else 'consensus'}.pkl"

if MODEL_BATCH_SIZE < 1 or MODEL_BATCH_SIZE * 128 > 65535:
    raise ValueError('MODEL_BATCH_SIZE must be between 1 and 511')

In [ ]:
from src.dataset import create_train_val_datasets
from src.masking import mask_slots
from src.training import build_initial_state
from src.utils import get_device

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Cannot load {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

checkpoint_dir = RUN_DIR / 'checkpoints'
run_config = load_module(f'evaluation_config_{RUN_DIR.name}', checkpoint_dir / 'used_config.py')
model_module = load_module(f'evaluation_model_{RUN_DIR.name}', checkpoint_dir / 'model_structure.py')
if DEVICE:
    device = torch.device(DEVICE)
else:
    try:
        device = get_device()
    except Exception:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset_path = Path(DATASET_PATH) if DATASET_PATH else Path(run_config.dataset_path)
steps = int(NB_OF_STEPS or run_config.NB_OF_STEPS)
checkpoint_path = checkpoint_dir / CHECKPOINT_NAME
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model = model_module.FlowSeparator(max_k=run_config.MAX_K, n_blocks=run_config.N_BLOCKS).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
model_accepts_mask = 'slot_mask' in inspect.signature(model.forward).parameters

_, val_dataset = create_train_val_datasets(
    dataset_path=dataset_path,
    train_size=run_config.TRAIN_SIZE,
    max_K=run_config.MAX_K,
    constant_K=run_config.CONSTANT_K,
    max_samples_train=0,
    max_samples_val=NUMBER_OF_SAMPLES,
    noise_train=True,
    noise_val=True,
    deterministic_train=False,
    deterministic_val=True,
    split_seed=run_config.SPLIT_SEED,
    split_strategy=run_config.SPLIT_STRATEGY,
    seed_train=run_config.SEED_TRAIN,
    seed_val=run_config.SEED_VAL,
    return_params=True,
)
dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
time_grid = 0.5 * (1 - torch.cos(torch.linspace(0.0, torch.pi, steps + 1, device=device)))
generator = torch.Generator(device=device).manual_seed(SEED)
print(f'Device: {device}')
print(f'Checkpoint: {checkpoint_path}')
print(f'Dataset: {dataset_path}')
print(f'Mixtures: {len(val_dataset)}, samplings: {NB_OF_SAMPLING}, steps: {steps}')

## Inference and alignment

In [ ]:
def predict_velocity(state, time, mixture, source_count, slot_mask):
    if model_accepts_mask:
        return model(state, time, mixture, source_count, slot_mask)
    return model(state, time, mixture, source_count)

def integrate_batch(mixture, target, source_count, slot_mask):
    initial_state = build_initial_state(
        mixture, target, source_count, slot_mask,
        nb_of_sampling=NB_OF_SAMPLING, generator=generator,
    )
    sampled_mixture = mixture.repeat_interleave(NB_OF_SAMPLING, dim=0)
    sampled_k = source_count.repeat_interleave(NB_OF_SAMPLING)
    sampled_mask = slot_mask.repeat_interleave(NB_OF_SAMPLING, dim=0)
    predictions = torch.empty_like(initial_state, device='cpu')
    for start in range(0, len(initial_state), MODEL_BATCH_SIZE):
        stop = min(start + MODEL_BATCH_SIZE, len(initial_state))
        state = initial_state[start:stop]
        local_mixture = sampled_mixture[start:stop]
        local_k = sampled_k[start:stop]
        local_mask = sampled_mask[start:stop]
        for step in range(steps):
            time = torch.full(
                (stop - start,), time_grid[step].item(),
                device=device, dtype=mixture.dtype,
            )
            velocity = predict_velocity(state, time, local_mixture, local_k, local_mask)
            state = mask_slots(
                state + (time_grid[step + 1] - time_grid[step]) * velocity,
                local_mask,
            )
        predictions[start:stop] = state.cpu()
    return predictions.reshape(len(mixture), NB_OF_SAMPLING, *target.shape[1:]).numpy()

def align_to_reference(prediction, reference):
    cost = np.mean((prediction[:, None] - reference[None]) ** 2, axis=(-1, -2))
    predicted_slots, reference_slots = linear_sum_assignment(cost)
    aligned = np.empty_like(prediction)
    aligned[reference_slots] = prediction[predicted_slots]
    assignment = tuple(zip(predicted_slots.tolist(), reference_slots.tolist()))
    return aligned, assignment

def iterative_consensus(predictions, max_iterations=20):
    provisional = np.stack([align_to_reference(item, predictions[0])[0] for item in predictions])
    provisional_mean = provisional.mean(axis=0)
    initial_medoid = int(np.mean((provisional - provisional_mean) ** 2, axis=(1, 2, 3)).argmin())
    reference = predictions[initial_medoid]
    previous_assignments = None
    for iteration in range(1, max_iterations + 1):
        aligned_items = [align_to_reference(item, reference) for item in predictions]
        aligned = np.stack([item[0] for item in aligned_items])
        assignments = tuple(item[1] for item in aligned_items)
        reference = aligned.mean(axis=0)
        if assignments == previous_assignments:
            return aligned, assignments, True, iteration
        previous_assignments = assignments
    return aligned, assignments, False, max_iterations

def apply_assignment(predictions, assignment):
    ordered = np.empty_like(predictions)
    for predicted_slot, reference_slot in assignment:
        ordered[:, reference_slot] = predictions[:, predicted_slot]
    return ordered

def global_medoid_index(aligned_sources):
    center = aligned_sources.mean(axis=0)
    distances = np.mean((aligned_sources - center) ** 2, axis=(1, 2, 3))
    return int(distances.argmin())

def effective_frequency_bounds(target_sources, relative_threshold=1e-2):
    energy = np.sum(target_sources.astype(np.float64) ** 2, axis=1)
    starts = []
    stops = []
    for source_energy in energy:
        maximum = float(source_energy.max())
        active = np.flatnonzero(source_energy >= relative_threshold * maximum) if maximum > 0 else np.array([], dtype=int)
        starts.append(int(active[0]) if len(active) else -1)
        stops.append(int(active[-1]) + 1 if len(active) else -1)
    return np.asarray(starts), np.asarray(stops)

def pairwise_spectral_coverage(starts, stops):
    intervals = [(int(start), int(stop)) for start, stop in zip(starts, stops) if start >= 0 and stop > start]
    if len(intervals) < 2:
        return 0.0
    values = []
    for index, (start, stop) in enumerate(intervals):
        for other_index, (other_start, other_stop) in enumerate(intervals):
            if index != other_index:
                overlap = max(0, min(stop, other_stop) - max(start, other_start))
                values.append(overlap / (stop - start))
    return float(np.mean(values))

In [ ]:
def prepare_result(mixture, target, predictions, k, snr, mixture_index):
    target_sources = target[:k]
    predicted_sources = predictions[:, :k]
    if ALIGN_TO_TARGET:
        aligned_items = [align_to_reference(item, target_sources) for item in predicted_sources]
        aligned_sources = np.stack([item[0] for item in aligned_items])
        assignments = tuple(item[1] for item in aligned_items)
        converged = True
        iterations = 1
        target_assignment = tuple((index, index) for index in range(k))
    else:
        aligned_sources, assignments, converged, iterations = iterative_consensus(predicted_sources)
        consensus = aligned_sources.mean(axis=0)
        _, target_assignment = align_to_reference(consensus, target_sources)
        aligned_sources = apply_assignment(aligned_sources, target_assignment)
    aligned_predictions = np.concatenate([aligned_sources, predictions[:, -1:]], axis=1).astype(np.float32)
    compact_target = np.concatenate([target_sources, target[-1:]], axis=0).astype(np.float32)
    medoid_index = global_medoid_index(aligned_sources)
    prediction_mean = aligned_predictions.mean(axis=0, dtype=np.float64).astype(np.float32)
    prediction_std = aligned_predictions.std(axis=0, dtype=np.float64).astype(np.float32)
    medoid_prediction = aligned_predictions[medoid_index]
    source_error = np.sum((aligned_sources - target_sources[None]) ** 2, axis=(-1, -2), dtype=np.float64)
    source_energy = np.sum(target_sources ** 2, axis=(-1, -2), dtype=np.float64)
    source_nmse = source_error / (source_energy[None] + 1e-12)
    source_nmse_db = 10 * np.log10(source_nmse + 1e-12)
    source_mse = np.mean((aligned_sources - target_sources[None]) ** 2, axis=(-1, -2), dtype=np.float64)
    residual_mse = np.mean((aligned_predictions[:, -1] - compact_target[-1]) ** 2, axis=(-1, -2), dtype=np.float64)
    total_mse = source_mse.mean(axis=1) + residual_mse
    uncertainty_rms = np.sqrt(np.mean(prediction_std[:k] ** 2, axis=(-1, -2), dtype=np.float64))
    medoid_rms = np.sqrt(np.mean(medoid_prediction[:k] ** 2, axis=(-1, -2), dtype=np.float64))
    relative_dispersion = uncertainty_rms / (medoid_rms + 1e-12)
    starts, stops = effective_frequency_bounds(target_sources)
    return {
        'mixture_index': int(mixture_index),
        'K': int(k),
        'mixture': mixture.astype(np.float32),
        'target': compact_target,
        'aligned_predictions': aligned_predictions,
        'prediction_mean': prediction_mean,
        'prediction_std': prediction_std,
        'medoid_prediction': medoid_prediction,
        'medoid_sampling_index': medoid_index,
        'snr': snr[:k].astype(np.float32),
        'assignments': assignments,
        'target_label_assignment': target_assignment,
        'consensus_converged': bool(converged),
        'consensus_iterations': int(iterations),
        'frequency_support_start': starts,
        'frequency_support_stop': stops,
        'frequency_overlap_pairwise': pairwise_spectral_coverage(starts, stops),
        'source_mse': source_mse.astype(np.float32),
        'residual_mse': residual_mse.astype(np.float32),
        'total_mse': total_mse.astype(np.float32),
        'source_nmse': source_nmse.astype(np.float32),
        'source_nmse_db': source_nmse_db.astype(np.float32),
        'uncertainty_rms': uncertainty_rms.astype(np.float32),
        'relative_dispersion_u': relative_dispersion.astype(np.float32),
    }

def summarize_by_k(results):
    rows = []
    for result in results:
        rows.append((result['K'], float(result['source_nmse'].mean()), float(result['relative_dispersion_u'].mean())))
    values = np.asarray(rows, dtype=np.float64)
    summary = {key: [] for key in ('K', 'count', 'nmse_mean', 'nmse_std', 'nmse_low', 'nmse_high', 'u_mean_percent', 'u_std_percent', 'u_low_percent', 'u_high_percent')}
    for k in sorted(np.unique(values[:, 0]).astype(int)):
        subset = values[values[:, 0] == k]
        nmse_mean = subset[:, 1].mean()
        nmse_std = subset[:, 1].std(ddof=1) if len(subset) > 1 else 0.0
        u_mean = 100 * subset[:, 2].mean()
        u_std = 100 * subset[:, 2].std(ddof=1) if len(subset) > 1 else 0.0
        for key, value in (
            ('K', k), ('count', len(subset)), ('nmse_mean', nmse_mean), ('nmse_std', nmse_std),
            ('nmse_low', max(0.0, nmse_mean - nmse_std)), ('nmse_high', nmse_mean + nmse_std),
            ('u_mean_percent', u_mean), ('u_std_percent', u_std),
            ('u_low_percent', max(0.0, u_mean - u_std)), ('u_high_percent', u_mean + u_std),
        ):
            summary[key].append(value)
    return {key: np.asarray(value) for key, value in summary.items()}

def make_curve(records, feature, metric, edges, scale=1.0, skip_k_one=False, min_samples=10):
    centers = 0.5 * (edges[:-1] + edges[1:])
    curves = []
    for k in sorted({record['K'] for record in records}):
        if skip_k_one and k == 1:
            continue
        subset = [record for record in records if record['K'] == k]
        feature_values = np.asarray([record[feature] for record in subset])
        metric_values = scale * np.asarray([record[metric] for record in subset])
        bin_ids = np.digitize(feature_values, edges[1:-1], right=False)
        point_x = []
        point_y = []
        point_count = []
        for bin_id, center in enumerate(centers):
            selected = metric_values[bin_ids == bin_id]
            if len(selected) >= min_samples:
                point_x.append(center)
                point_y.append(selected.mean())
                point_count.append(len(selected))
        point_x = np.asarray(point_x)
        point_y = np.asarray(point_y)
        if len(point_x) >= 3:
            curve_x = np.linspace(point_x.min(), point_x.max(), 250)
            curve_y = PchipInterpolator(point_x, point_y)(curve_x)
        else:
            curve_x = point_x.copy()
            curve_y = point_y.copy()
        curves.append({
            'K': int(k), 'point_x': point_x, 'point_y': point_y,
            'point_count': np.asarray(point_count), 'curve_x': curve_x, 'curve_y': curve_y,
        })
    return curves

def build_plot_data(results):
    mixture_records = []
    source_records = []
    for result in results:
        mixture_records.append({
            'K': result['K'],
            'overlap': result['frequency_overlap_pairwise'],
            'nmse': float(result['source_nmse'].mean()),
            'u': float(result['relative_dispersion_u'].mean()),
        })
        for source in range(result['K']):
            source_records.append({
                'K': result['K'],
                'snr': float(result['snr'][source]),
                'nmse': float(result['source_nmse'][:, source].mean()),
                'u': float(result['relative_dispersion_u'][source]),
            })
    snr_values = np.asarray([record['snr'] for record in source_records])
    snr_edges = np.linspace(snr_values.min(), snr_values.max(), 19)
    overlap_edges = np.linspace(0.0, 1.0, 19)
    return {
        'by_k': summarize_by_k(results),
        'curves': {
            'nmse_by_snr': make_curve(source_records, 'snr', 'nmse', snr_edges),
            'nmse_by_overlap': make_curve(mixture_records, 'overlap', 'nmse', overlap_edges, skip_k_one=True),
            'u_by_snr': make_curve(source_records, 'snr', 'u', snr_edges, scale=100.0),
            'u_by_overlap': make_curve(mixture_records, 'overlap', 'u', overlap_edges, scale=100.0, skip_k_one=True),
        },
        'snr_edges': snr_edges,
        'overlap_edges': overlap_edges,
        'minimum_bin_count': 10,
    }

## Execution and saving 

In [ ]:
results = []
mixture_offset = 0
generator.manual_seed(SEED)
with torch.inference_mode():
    for mixture, target, source_count, slot_mask, params in tqdm(dataloader, desc='Euler sampling'):
        mixture_device = mixture.to(device)
        target_device = target.to(device)
        source_count_device = source_count.to(device)
        slot_mask_device = slot_mask.to(device)
        batch_predictions = integrate_batch(mixture_device, target_device, source_count_device, slot_mask_device)
        mixture_values = mixture.numpy()
        target_values = target.numpy()
        source_counts = source_count.numpy()
        snr_values = params['snr'].numpy()
        for index in range(len(mixture)):
            results.append(prepare_result(
                mixture_values[index], target_values[index], batch_predictions[index],
                int(source_counts[index]), snr_values[index], mixture_offset + index,
            ))
        mixture_offset += len(mixture)

plot_data = build_plot_data(results)
checkpoint_epoch = int(checkpoint.get('epoch', -1))
payload = {
    'schema_version': 1,
    'metadata': {
        'run_dir': str(RUN_DIR),
        'checkpoint': str(checkpoint_path),
        'checkpoint_epoch': checkpoint_epoch + 1 if checkpoint_epoch >= 0 else None,
        'dataset': str(dataset_path),
        'number_of_samples': len(results),
        'batch_size': BATCH_SIZE,
        'model_batch_size': MODEL_BATCH_SIZE,
        'number_of_samplings': NB_OF_SAMPLING,
        'number_of_steps': steps,
        'seed': SEED,
        'device': str(device),
        'solver': 'euler',
        'time_grid': time_grid.cpu().numpy(),
        'alignment': 'target' if ALIGN_TO_TARGET else 'iterative_consensus',
        'medoid': 'single sampling minimizing squared distance to the aligned source mean',
        'relative_dispersion_u': 'sampling standard-deviation RMS divided by the matching global-medoid source RMS',
        'frequency_support_relative_threshold': 1e-2,
        'frequency_overlap_pairwise': 'mean ordered-pair spectral support coverage',
    },
    'results': results,
    'plot_data': plot_data,
}
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('wb') as file:
    pickle.dump(payload, file, protocol=pickle.HIGHEST_PROTOCOL)
print(f'Saved {len(results)} mixtures to {OUTPUT_PATH}')